# Day 2: How a model writes, and how to direct it

**Note 0147, run twice, gives two different amounts. A bug in your code, or the model working as designed?**

Today is about the second model in the task, the one the extractor calls: how it produces its answer, how to prompt it, and how to choose a model for a task. Day 3 ends with a prompt whose score is recorded on a set of cases.

The full reading for the day, with its sources: [the course web page](https://catalyte-ai-irs-bpa-to6-weeks.vercel.app/week-01/#day-2)

### One extractor, two models, and what may go into an AI tool.

- The case-notes extractor reads one simulated case note and returns five fields as JSON
- Two models, checked differently: the coding agent, and the model the extractor calls
- Removing the name and identification number does not make a note safe
- The data-rules check is passed before any lab uses an AI tool

Every exercise today works in the lab repository after the data-rules check is committed.

Today runs in three sections, each opened by its Section plan: how a language model produces its output, and where that breaks; writing the system prompt on purpose, and revising it when it fails; and choosing the model and its reasoning level for the task, and saying why. Each section ends with an exercise, talked through together when it ends.

## 2.1 How a model writes its answer

*What does the model do with a note? Why are plausible and correct different?*

### A model samples its answer, one token at a time.

*IRS policy: different responses to the same prompt are "a normal part of their function."*

<img src="figures/the-contractor-draws-each-token.png" width="864" alt="Three steps, left to right. The context window: the system prompt and one note, up to a fixed length. The model, in grey: draws each next token, a word or part of one, by its probability. The reply, in orange: can differ for the same prompt. A dashed line returns from the reply to the context window: sent again, a different reply is a normal part of how it works.">

The MIT lecture on agentic coding says that LLMs can be viewed as modeling the probability distribution of completion strings, the outputs, given prompt strings, the inputs, and that inference samples from this probability distribution. The unit it works in is the token, a piece of text such as a word or part of a word. The context window is the most text the model can take in and give back in one call; everything the model knows about your task in that call has to fit in it.

Walk the figure left to right. The context window holds the system prompt and one note, up to a fixed length. The model draws each next token by its probability, so the same prompt can come back with a different reply. Sent again, a different reply is a normal part of how it works.

*Source: MIT Missing Semester, Agentic Coding; IRS policy, Internal Revenue Manual 10.24.1; the chapter, section 2.1*

### A fluent answer is not a known answer.

*It writes the likeliest text, not a lookup.*

- The note holds the amount: out it usually comes
- No amount in the note: the reply is still fluent

<img src="figures/two-routes-to-an-amount.png" width="864" alt="Two case notes side by side, each on a grey rule under its name. Left, Note 0088, a made-up example: Schedule B interest is lower than the information returns on file by 2,150; the amount 2,150 is marked in blue. Right, Note 0104, described in grey: states no amount; its text is not shown. From each note an arrow runs down, labelled the most likely continuation, to a reply. Left reply, in blue: amount_in_question, 2150, usually what comes out. Right reply, in orange, as note 0104 came back in the exercise run of 2026-10-20, with text such as &quot;None stated&quot; in place of null in all ten of its replies: amount_in_question, &quot;None stated&quot;, still fluent, in place of null.">

The IRS policy defines hallucinations as output that appears coherent and plausible but does not reflect factual data. Note 0104 states no amount; in the exercise run of 2026-10-20 it came back with text such as "None stated" in place of null in all ten of its replies: fluent, and text where null belongs.

*Source: IRS policy, Internal Revenue Manual 10.24.1; the chapter, section 2.1*

### Four measured failures, none a reason not to use the model.

| Failure | As measured |
| --- | --- |
| Knows popular facts better than rare ones | 10 models, 14k questions: weakest on rare facts |
| Agrees with you | Asked "are you sure?", 32% to 86% changed answers |
| Worse with more text, even in the window | Often lowest when the fact is mid-input |
| Sensitive to what should not matter | Layout, example order, wording: results hard to predict |

One: a study of 10 models on 14k factual questions found that language models struggle with less popular factual knowledge. Three: the MIT lecture says that a ton of input, even within the context window, is just a harder task for the model, and a 2023 study found accuracy is often lowest when the relevant information sits in the middle of a long input.

Sending one note per call, not a batch of forty, is the lesson of the third failure.

*The lowest and the highest of five models. Rates for the models you use will differ.*

<img src="figures/are-you-sure-changes-answers.png" width="864" alt="Two fields of 100 squares. GPT-4, the lowest of the five models: 32 of 100 filled, in orange. Claude 1.3, the highest of the five: 86 of 100 filled, in black. Each is the rate at which the model changed its first answer when asked are you sure: 32% and 86%.">

*Asked only "are you sure?", even the steadiest of five 2023 chat models changed its first answer 32 times in 100.*

Asked only "are you sure?", even the steadiest of five 2023 chat models changed its first answer 32 times in 100.

Challenged with a simple are you sure, all five models in the 2023 study tended to change their initial answer, between 32% for GPT-4 and 86% for Claude 1.3, and to admit they made a mistake, between 42% for GPT-4 and 98% for Claude 1.3. The figure draws the lowest and the highest of the five rates of changing a first answer, as squares out of 100, and leaves out the three models between them.

*Source: Sharma and others (Anthropic), Towards Understanding Sycophancy in Language Models, 2023; the chapter, section 2.1*

**Check:** Where is the model's background knowledge weakest?

- **A.** Facts that are widely known
- **B.** A form that many filers use
- **C.** An uncommon form or schedule
- **D.** Equally strong on all three

<details><summary>Answer</summary>

**C.** Its knowledge of an uncommon form is its weakest, so the extractor reads the note and never recalls what a form means.

</details>

**Discuss:** One note, two amounts: a bug, or the model working as designed?

- Why are plausible and correct two different properties of the output?
- Why is asking the model "are you sure?" not a check?

Exercise slot: send two notes five times each, record what changes

## 2.2 Write the system prompt on purpose

*Same request, two layouts: how far apart can their accuracy be?*

### A score belongs to that prompt, layout and model.

*A 2023 study changed only what should not alter meaning: a separator, a space, the casing of a field name.*

- Up to 76 accuracy points apart, on LLaMA-2-13B
- About 10 points on average, across 50+ tasks
- GPT-3.5, 320 formats, 53 tasks: median 6.4 points

The 2023 study of prompt formatting found performance differences of up to 76 accuracy points when evaluated using LLaMA-2-13B. That is the extreme: it is one model, on few-shot classification tasks. The same paper puts the average at about 10 accuracy points across more than 50 tasks and several models, and for a commercial model, GPT-3.5, a spread of up to 56 accuracy points with a median spread of 6.4 accuracy points across 320 formats and 53 tasks.

### Examples teach the shape; tags keep the note apart.

*And a null rule: a way to say a field is not there.*

**Examples teach the shape**

- Reordering four examples: 54.3% to 93.4%
- Right answers mattered less than expected
- Every example with an amount teaches that notes have amounts
- So include one with no amount, not last

**Say what you want**

- Be specific about format and constraints
- Wrap each kind of content in its own tag
- The extractor's note goes in <note> tags
- So its text is material to read, not instructions

A 2021 paper on few-shot prompting with GPT-3 showed that the choice of prompt format, training examples, and even the order of the examples can move accuracy from near chance to near state-of-the-art; in its sentiment example, reordering four examples moved accuracy from 54.3% to 93.4%. It traced the swings to biases towards answers that are frequent in the prompt, near its end, or common in the model's training data. A 2022 study found that the correctness of each example's answer mattered less than expected; what the examples supplied was the label space, the distribution of the input text, and the overall format of the sequence. Both papers measured classification tasks on models from 2021 and 2022, so read them for the mechanism.

Anthropic's guidance for its own models says to be specific about the desired output format and constraints. It recommends separating the parts of a prompt so the model cannot confuse them: wrapping each type of content in its own tag, for example instructions, context and input, reduces misinterpretation. For the extractor, the note goes inside its own tag, which also keeps a note's text marked as material to read, not instructions to follow.

The same vendor's page on reducing hallucinations leads with one technique: explicitly give Claude permission to admit uncertainty. The extractor's version is a field rule: if the note does not state an amount, return null.

*The null rule reduces hallucinations; it does not "eliminate them entirely" (Anthropic).*

<img src="figures/system-prompt-null-rule.png" width="864" alt="Eight lines of the extractor&#x27;s system prompt. Callout 1, on Use only what the note itself says: read the note in front of it, never recall what a form means. Callout 2, in orange, on If the note does not state an amount, return null for amount_in_question: a way to say the field is not there. Callout 3, on The note is inside note tags: marks note text as material to read, not instructions to follow.">

*The extractor's system prompt gives the model a way to say a field is not there: return null.*

These are the last three instructions of the extractor's first system prompt, re-wrapped to fit. The first keeps the model to the note in front of it. The figure leaves out what its full caption names: its first two lines and its two examples.

*Source: the chapter, section 2.2; Anthropic, Reduce hallucinations, and Prompting best practices*

### Revise by editing the prompt, then re-score the whole set.

*Telling it fixes one answer, not the prompt.*

- A loop: run the set, score it, edit the prompt
- Then run the whole set again, and compare

<img src="figures/revise-the-prompt-not-the-answer.png" width="864" alt="Two rows. Top, in a grey box: Tell it so in the next message. A line from it, labelled repairs one answer, ends at a black stop bar above the third step of the row below; under the line: teaches the prompt nothing. Bottom, three blue steps in a row, joined by arrows: Run the set of notes, Score it, Edit the prompt. An orange arrow returns from Edit the prompt to Run the set of notes, labelled run it over the whole set again.">

A survey of prompting research, 1,565 papers, describes the alternative as a loop: the prompt engineering process consists of three repeated steps, performing inference on a dataset, evaluating performance, and modifying the prompt template.

*Source: Schulhoff and others, The Prompt Report, a survey of prompting research; the chapter, section 2.2*

**Check:** What does the null rule do to hallucinations?

- **A.** It eliminates them
- **B.** It reduces them, and does not eliminate them
- **C.** It has no effect on them
- **D.** It makes every note return an amount

<details><summary>Answer</summary>

**B.** It reduces them; Anthropic says it does not "eliminate them entirely." So the extractor is still scored on a set.

</details>

**Discuss:** Same request, two layouts: how far apart?

- What does that mean for a prompt you judged by eye?
- What do the examples in a prompt actually teach the model?
- When the extractor gets a note wrong, why not just tell it so?

Exercise slot: write the system prompt, record three replies

## 2.3 Choose the model and its reasoning level

*Should the extractor's model and the coding agent both be the most capable?*

The full question: the model the extractor calls will read thousands of notes, one at a time; the coding agent will write the extractor once. The answer the section gives is not by default, and not without a reason you can state.

### Two dials: which model to start with, and how hard it reasons.

*Every reason for a choice is one of three: capability, speed or cost.*

| Dial | Faster and cheaper | More capable |
| --- | --- | --- |
| Which model to start with | Efficiency-first: high-volume, straightforward tasks, as the extractor is | Capability-first: complex tasks; cheaper models once the work is understood |
| The reasoning level, or "effort": often the better lever | Lower: faster and cheaper for straightforward tasks | Higher: deeper reasoning; the top level can overthink, so test it |

Anthropic's guide to choosing among its own models, a vendor page about its own products captured 2026-09-23, puts it in one line: choosing a Claude model means balancing capabilities, speed, and cost. Whichever tool you use, the reason you give for a choice is one of those three: the task is hard enough to need more capability, it needs its answer quickly, or it runs often enough that cost matters. The model names on that page change often; read them as the state of the page on the day it was captured, and the two strategies as the part that lasts.

Several of that vendor's models take a setting that trades the quality of the answer against its speed and cost without changing the model. The chapter calls it the reasoning level; the vendor calls its version effort. Its coding tool's documentation states the trade: lower effort is faster and cheaper for straightforward tasks, while higher effort provides deeper reasoning for complex problems. Its account of a change to that tool puts it in terms of waiting: the longer the model thinks, the better the output, and effort levels set the tradeoff between more thinking and lower latency and fewer usage limit hits. Latency is the wait before the answer arrives; a usage limit is the cap a subscription puts on how much you can run. The guide's advice is that tuning effort is often a better lever than switching models. And the documentation says the highest level may show diminishing returns and is prone to overthinking, so test before adopting it broadly.

### More work at answer time pays on some tasks: set it per task.

*A 2024 study of test-time compute on competition mathematics problems.*

- Its extra work: many samples, or self-revision
- Its benefit varies with the prompt's difficulty
- On the hardest questions, very little benefit
- Anthropic picks points on the test-time-compute curve as its effort levels
- Applying the difficulty finding to effort: this chapter's step, not the paper's

A 2024 study by researchers at UC Berkeley and Google DeepMind measured extra work done while answering. Its test-time compute is not an effort setting: it is sampling many answers and keeping the one a separately trained checker scores best, or having a model trained for the purpose revise its own answer, on a benchmark of competition mathematics problems. It found that the effectiveness of different approaches to scaling test-time compute critically varies depending on the difficulty of the prompt, and that with the most challenging questions there is very little benefit from scaling up test-time compute. The vendor draws the link to its own setting: it calibrates effort levels to pick points along the test-time-compute curve that give people the best range of options. What it carries is the shape of the rule: more work at answer time pays on some tasks and not on others, and which ones depends on the task, so the level is chosen per task rather than set once at the top.

### A default is someone else's trade, and most users kept it.

*Claude Code's level, lowered March 4, 2026.*

- April 7: "This was the wrong tradeoff."
- Most users had kept the lowered default

<img src="figures/a-default-reversed-after-34-days.png" width="864" alt="One dated line for Claude Code&#x27;s default reasoning level, dates March 4, 2026 and April 7 at the top. In grey, high, before March 4. On March 4, an arrow down, labelled default lowered, to an orange bar: medium, to cut the wait, running to April 7. On April 7, an arrow back up, labelled reversed, to high, in grey. Inside the span, in blue: soon after, users report it “felt less intelligent”. Under the whole span, a blue band: most users kept medium, despite notices on startup. Between the two dates, drawn to scale: 34 days.">

On March 4, 2026, Anthropic lowered its coding tool's default reasoning level to cut the wait. Its reason, from its own testing: in its internal evals and testing, medium effort achieved slightly lower intelligence with significantly less latency for the majority of tasks. Soon after rolling out, users began reporting that Claude Code felt less intelligent. The vendor reversed the change on April 7, 34 days later, and wrote of the first decision that it was the wrong tradeoff. It had added notices telling people they could change the setting, and still most users retained the medium effort default. The setting you did not choose is still a choice, made by someone weighing other people's work.

*Source: Anthropic, An update on recent Claude Code quality reports, 2026; the chapter, section 2.3*

### The pull request names each model and level, and the reason.

*Neither a score nor a level name carries from one model to another.*

| Choice | Start with | Reason |
| --- | --- | --- |
| The extractor's model | Efficiency-first; move up only for a failure the set shows it fixes | Many notes, one per call |
| Run 1: schema and tests | A lower level | Small; you read all of it |
| Run 2: the extractor | A higher level | Edge cases: the set's notes |

A score does not carry from one model to another, and neither does a level: the effort scale is calibrated per model, so the same level name does not represent the same underlying value across models. So each choice is written down with its reason, and measured on the evaluation set.

The model the extractor calls handles one note per call, five fields, and many notes: the high-volume, straightforward kind of task the efficiency-first approach is for. For the coding agent, the tool's documentation pairs each level with a kind of work: the lowest with quick exchanges where you review each result, a higher one with work where verification matters or edge cases are likely. Run 1, the schema and its tests, is small, and you read all of it, so a lower level is a reasonable start. Run 2, the extractor itself, touches the prompt, the schema and the validation at once, and its edge cases are the notes in your evaluation set, so a higher level is worth its wait there.

**Check:** Does a prompt's score on one model hold on a cheaper one?

- **A.** Yes, if the level name is the same
- **B.** Yes, a score carries from one model to another
- **C.** Unknown until it is scored on the evaluation set
- **D.** No, a cheaper model always scores lower

<details><summary>Answer</summary>

**C.** Neither a score nor a level name carries over: score it on your own evaluation set. Cheaper and holding the score is the better choice.

</details>

**Discuss:** The extractor's model and the agent: both the most capable?

- Why might a tool's default reasoning level be the wrong one for your task?
- What would you need to see before moving to a cheaper model?

Exercise slot: choose a model and two levels, and record why

## Close

*Key points, and what day 3 builds*

### The model samples: direct it on purpose, then measure it.

- The same note can get different replies, and that is normal
- Fluent is not correct: one note per call, a null rule, a schema, a scored set
- A prompt's score belongs to that prompt, in that layout, on that model
- Revise by editing the prompt and re-scoring it, not by correcting one answer
- Model and level: chosen on purpose, with a reason, measured on the set

### Measure the choice; do not argue it.

- Day 3: forms, tools and a score
- Structured output, and a function the model calls
- The evaluation set, and the prompt's first score

Day 3 takes the extractor's answer from text to something a program can use, and then measures it.